# 04 - JOB-02..07: one resumable training run (gated: train_main = B1-B12, D1-D6)

**Private compute session - not a dataset mirror.** Raw BraTS data come only from the official TCIA source into ephemeral session storage (`WORK`) and are never written to notebook output, a Kaggle Dataset, GitHub or the website. B1 basis: owner-approved alternative (amendment v1.0-A1); no TCIA authorization is claimed. Enable **GPU** and **Internet** in the notebook settings. Never paste credentials into a cell; use Kaggle Secrets if a private repository needs a token.

In [ ]:
# PARAMETERS - set before running (no secrets here)
REPO_URL = ""   # https URL of this research repository (GitHub)
COMMIT = ""     # exact commit SHA or tag to run (never a moving branch)
WORK = "/tmp/brats"                     # ephemeral session storage (raw data stay here)
EXPORT = "/kaggle/working/export"       # records/reports/artifacts only - never raw data
assert REPO_URL, "set REPO_URL"
assert COMMIT, "set COMMIT"


In [ ]:
import os
import subprocess

os.makedirs(WORK, exist_ok=True)
os.makedirs(EXPORT, exist_ok=True)
!git clone --quiet {REPO_URL} /kaggle/working/repo
%cd /kaggle/working/repo
!git checkout --quiet {COMMIT}
!git rev-parse HEAD
!pip install --quiet -e ".[io,nnunet]"
!brats-uncertainty verify-protocol

In [ ]:
r = subprocess.run(["brats-uncertainty", "check-action", "train_main"], capture_output=True, text=True)
print(r.stdout)
assert r.returncode == 0, "gate not open for train_main: stop here"

In [ ]:
JOB = ""             # JOB-02 ... JOB-07 (arm/seed from remote_compute.yaml)
DATASET_ID = None    # nnU-Net dataset id
DATASET_NAME = ""    # e.g. BraTS2021Dev
MANIFEST_SHA256 = ""  # committed B5 manifest_sha256
SPLIT_SHA256 = ""     # committed B12 split hash
CASE_IDS = ""         # file with the frozen train+validation IDs (B10)
RESUME = False       # True only to continue this run's own checkpoint
RESULTS_ROOT = "/kaggle/working/nnunet_results"  # run dirs persist as output
assert JOB
assert DATASET_ID
assert MANIFEST_SHA256
assert SPLIT_SHA256

1. Re-acquire the official files (notebook 01) and run `verify-inventory` against the committed B2 record.
2. Build the nnU-Net raw dataset with provenance, then let nnU-Net plan/preprocess.
3. To resume after a session ended: attach the previous output, copy `nnunet_results/` back, set `RESUME = True`.

In [ ]:
RAW = f"{WORK}/nnUNet_raw/Dataset{DATASET_ID:03d}_{DATASET_NAME}"
os.environ["nnUNet_raw"] = f"{WORK}/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = f"{WORK}/nnUNet_preprocessed"
if not os.path.isdir(RAW):
    !brats-uncertainty build-nnunet-dataset --training-root {WORK}/raw/RSNA-ASNR-MICCAI-BraTS-2021/BraTS2021_TrainingSet --out {RAW} --dataset-id {DATASET_ID} --dataset-name {DATASET_NAME} --case-ids {CASE_IDS}
    !nnUNetv2_plan_and_preprocess -d {DATASET_ID} -c 3d_fullres --verify_dataset_integrity

In [ ]:
RESUME_FLAG = "--resume" if RESUME else ""
!brats-uncertainty compute-preflight --job {JOB} --work-dir {WORK}
!brats-uncertainty job-run {JOB} --dataset-id {DATASET_ID} --results-root {RESULTS_ROOT} --dataset-provenance {RAW}/conversion_provenance.json --manifest-sha256 {MANIFEST_SHA256} --split-sha256 {SPLIT_SHA256} {RESUME_FLAG}